# Day 08 · 하네스 — 에이전트의 작업 환경

7강까지는 **손으로** 에이전트를 만들었다. 오늘부터는 **실제 코드 에이전트(Claude Code · Codex)를 켜고** 그 작업 환경을 세팅한다.

> 같은 모델도 **하네스가 결과를 가른다.**

**오늘의 실습 3개** — 모두 터미널·Claude Code에서 진행한다 (파이썬 실행 셀 없음).

| 랩 | 무엇을 | 결과물 |
|---|---|---|
| Lab 1 | buggy-request 레포로 **CLAUDE.md 작성** | 내 프로젝트 룰 파일 |
| Lab 2 | 같은 레포에서 **심어둔 결함 찾기** | 회수율 측정표 |
| Lab 3 | **슬랙을 MCP로 연결** | 말 한마디로 슬랙 게시 |

**사전 준비** — Claude Code 설치·로그인 (Pro/Max/Team/Console 플랜 필요, 무료 플랜 불가). 설치는 `day08/Claude_Code_Codex_세팅.md` 참고.


## Lab 1 · buggy-request 레포로 CLAUDE.md 작성

완벽한 룰 파일이 목표가 아니다. **초안을 빠르게 뽑고, 실수마다 한 줄씩 추가**하는 게 정석이다.

### 1. 클론 → /init

```bash
git clone https://github.com/TunaLee/buggy-request.git
cd buggy-request
claude
```

```
> /init
  코드베이스 분석 → CLAUDE.md 초안 자동 생성
```

### 2. /memory 로 로드 확인

```
> /memory
  로드된 파일 목록 · 자동 메모리 ON/OFF
```

목록에 CLAUDE.md가 **없으면** 위치·파일명 문제다 — Claude가 아예 못 본 것.

### 3. 5섹션으로 정리

```markdown
# 스택
Next.js 15 (App Router) · TypeScript · Prisma + Postgres

# 디렉토리
src/app/   페이지·API
prisma/    스키마·migration
tests/     vitest 단위·통합

# 명령어
npm run dev · npm test · npx prisma migrate dev

# 룰
- server component 우선 (use client 최소화)
- import 절대경로 @/...
- 새 API는 zod로 입력 검증

# 하지 말 것
- .env / .env.local 커밋
- any 사용
- 테스트 없이 라우트 추가
```

### 4. 같은 태스크를 전/후로 비교

`"테스트 1개 추가해줘"` 를 **CLAUDE.md 없이 / 있이** 각각 시켜본다. 무엇이 달라지는가?

### 5. 빗나간 부분 → 룰 한 줄로 환원

- `"any 썼네"` → **하지 말 것**에 추가
- `"테스트 위치 틀렸네"` → **디렉토리**에 추가

### 막혔을 때

한 줄도 안 떠오르면 — **최근 에이전트가 가장 짜증나게 했던 실수 1개**만 떠올려 "하지 말 것"에 적는다. 거기서부터 거꾸로 채워진다.

### 원리

CLAUDE.md는 **강제가 아니라 컨텍스트**다. 따르려고 읽는 것일 뿐, 반드시 막아야 하면 **Hook**으로 (9강).

마지막에 `git commit` — 다음 세션부터 **프롬프트 캐싱**이 걸린다.


## Lab 2 · 심어둔 결함 찾기 (회수율 측정)

Lab 1에서 클론한 **그 buggy-request 레포 그대로**. 곳곳에 결함이 숨어 있다 — 보안 · 로직 · 런타임.

작성한 CLAUDE.md를 컨텍스트로, **어떤 도구 조합이 결함을 얼마나 잡아내는지 직접 측정**한다.

### 측정 절차 — 변수를 하나씩만 바꾼다

| 단계 | 명령 | 찾은 결함 수 |
|---|---|---|
| ① | `/review` 단독 | ___ 개 |
| ② | `+ /cso` (보안 특화) | ___ 개 |
| ③ | `+ 테스트 · /qa` (런타임) | ___ 개 |

```
> /review
  ... 발견 항목 기록

> /cso
  보안 특화 점검 — 회수율 변화 확인

> /qa
  런타임에서만 드러나는 결함 (off-by-one · 상태 전이 · 삭제 미반영)
```

### 왜 이렇게 하나

어떤 도구·조합이 회수율을 끌어올리는지는 **감이 아니라 숫자로** 확인한다. 변수를 하나씩만 바꿔 재측정하면 차이가 드러난다.

### 리스크 티어 → 명령 매핑

| 리스크 | 명령 |
|---|---|
| 낮음 | `/code-review` · `/qa` |
| 중간 | + `/security-review` · `/review` |
| 높음 | + `/cso` · **사람 최종 승인** |

### 원리

에이전트는 **제안이 아니라 실행**한다. 삼성(데이터 유출) · EchoLeak(프롬프트 인젝션) · Replit(폭주) — 셋 다 나중에 막을 문제가 아니라 **설계에서 막을 문제**였다.

> 이 회수율 측정 방식이 곧 **도입 보고서의 근거**가 된다 (9강 도입 로드맵).


## Lab 3 · 슬랙을 MCP로 연결

사내 시스템을 에이전트에 잇는다. **노출한 도구가 곧 보안 경계**다.

### 1. Slack 앱 생성

`api.slack.com` → **Create an App** → From scratch (앱 이름 · 워크스페이스 선택)

### 2. OAuth 스코프 추가

**OAuth & Permissions** → Bot Token Scopes 에 `chat:write` 등 추가

### 3. 워크스페이스에 설치 → Allow

스코프를 넣으면 **Install to Workspace**가 활성화된다. 허용하면 토큰 발급.

### 4. 두 값 복사

- **Bot User OAuth Token** (`xoxb-…`) → `SLACK_BOT_TOKEN` — **비밀값이다. 마스킹·외부 노출 금지**
- 주소창 URL의 첫 ID (`T…`) → `SLACK_TEAM_ID`

### 5. MCP 등록

```bash
claude mcp add slack \
  -e SLACK_BOT_TOKEN=xoxb-… \
  -e SLACK_TEAM_ID=T… \
  -- npx -y @modelcontextprotocol/server-slack
```

재시작 후 `/mcp` 로 `slack connected` 확인. 봇을 채널에 `/invite` 한다.

### 6. 말 한마디로 게시

```
> #dev 채널에 오늘 변경사항 요약해서 올려줘
  slack.post_message
```

### 왜 봇 토큰인가

공식 서버(`mcp.slack.com`)는 Claude Code의 자동 인증(DCR)을 받지 않아 **인증에 실패한다**. 그래서 봇 토큰으로 붙인다.

### 트러블슈팅 — `-32000`

서버가 떴다 **즉시 종료**됐다는 뜻(Connection closed). 원인은 거의 **env · 패키지 · 윈도우 spawn**이다.

```bash
# 1. 진짜 에러 보기 — Claude는 stderr를 숨긴다
SLACK_BOT_TOKEN=xoxb-… SLACK_TEAM_ID=T… npx -y @modelcontextprotocol/server-slack
claude --debug

# 2. 등록값 확인 — 토큰·TeamID 오타가 가장 흔하다
claude mcp get slack

# 3. Windows — cmd /c 로 감싸 stdio 즉시 종료 방지
claude mcp add slack -e … -- cmd /c npx -y @modelcontextprotocol/server-slack
```

### 원리

MCP = **웨이터**다. 머리는 Claude, 손발은 MCP 서버. **메뉴에 없는 건 못 시킨다** — 노출한 도구만 호출되니 그게 곧 보안 경계다.

> 5강에서 만든 우리 MCP 서버도 **똑같은 방식**으로 붙는다: `claude mcp add day07-tools -- python server.py`


## Lab 4 · 앱 뼈대 스크래치 빌드 — MCP 호스트

**보일러플레이트 없이 맨바닥부터.** 오늘 끝나면 브라우저에서 **Qwen과 대화**된다.

> 직접 타이핑하지 않는다. **Claude Code에게 시키고, 내가 판정한다.**

---

### 사전 준비 두 가지

**1. Neon DB** — [neon.tech](https://neon.tech) 가입 → 프로젝트 생성 → **Connection string** 복사
```
postgresql://user:pw@ep-xxx.ap-southeast-1.aws.neon.tech/neondb?sslmode=require
```

**2. NVIDIA API 키** — 7강에서 쓰던 `nvapi-...` 그대로

---

### Step 1 · 프로젝트 생성

```bash
npx create-next-app@latest mcp-host --typescript --tailwind --app --no-src-dir
cd mcp-host
npm install openai prisma @prisma/client
npx prisma init --datasource-provider postgresql
claude
```

Claude Code를 켜고 **먼저 `/init`** 으로 CLAUDE.md를 만든다 (Lab 1에서 한 그대로).

### Step 2 · 환경변수 — 키는 서버에만

`.env` (git에 **절대** 안 올린다):

```bash
DATABASE_URL="postgresql://...neon.tech/neondb?sslmode=require"
NVIDIA_API_KEY="nvapi-xxxxxxxx"
```

`.gitignore` 에 `.env` 가 있는지 **반드시 확인**한다.

> `NEXT_PUBLIC_` 접두사가 붙은 것만 브라우저에 노출된다. **키에는 절대 붙이지 않는다.**

### Step 3 · Prisma 스키마 → Neon에 실제 테이블

Claude Code에 시킨다:

```
prisma/schema.prisma 에 대화 저장용 모델을 만들어줘.

- Conversation: id(cuid), title(기본 "새 대화"), createdAt, messages 관계
- Message: id(cuid), role(user|assistant|tool), content, createdAt,
           conversationId → Conversation 관계 (onDelete: Cascade)

그다음 npx prisma migrate dev --name init 을 실행해서
Neon에 실제 테이블을 만들어줘.
```

**확인** — Neon 콘솔의 Tables 탭에 `Conversation`·`Message` 가 보이면 성공이다.

### Step 4 · API Route — NVIDIA Qwen 스트리밍

```
app/api/chat/route.ts 를 만들어줘.

- POST로 { messages } 를 받는다
- NVIDIA API (baseURL: https://integrate.api.nvidia.com/v1) 로 Qwen 을 부른다
  model: qwen/qwen3-next-80b-a3b-instruct
- 키는 process.env.NVIDIA_API_KEY — 절대 클라이언트로 보내지 마
- stream: true 로 받아서 ReadableStream 으로 토큰을 흘려보낸다
```

나와야 할 뼈대:

```typescript
import OpenAI from "openai";

const llm = new OpenAI({
  baseURL: "https://integrate.api.nvidia.com/v1",
  apiKey: process.env.NVIDIA_API_KEY,      // 서버 전용
});

export async function POST(req: Request) {
  const { messages } = await req.json();

  const stream = await llm.chat.completions.create({
    model: "qwen/qwen3-next-80b-a3b-instruct",
    messages,
    stream: true,
    temperature: 0.2,
  });

  const encoder = new TextEncoder();
  return new Response(
    new ReadableStream({
      async start(controller) {
        for await (const chunk of stream) {
          const delta = chunk.choices[0]?.delta?.content ?? "";
          if (delta) controller.enqueue(encoder.encode(delta));
        }
        controller.close();
      },
    }),
    { headers: { "Content-Type": "text/plain; charset=utf-8" } }
  );
}
```

### Step 5 · 채팅 UI — 스트리밍을 받는다

```
app/page.tsx 를 채팅 UI로 만들어줘.

- "use client" — 입력을 받아야 하니 Client Component
- 메시지 목록 + 입력창 + 전송 버튼
- /api/chat 으로 POST 하고, 스트림을 reader로 읽어
  글자가 타이핑되듯 붙게 해줘
- Tailwind로 간단히
```

### Step 6 · 확인

```bash
npm run dev     # http://localhost:3000
```

- [ ] 질문을 넣으면 **글자가 흘러나온다** (한 번에 안 나온다)
- [ ] 개발자도구 Network 탭에서 **API 키가 안 보인다**
- [ ] Neon 콘솔에 테이블이 있다

---

### 판정 — 내가 짠 것과 비교한다

| 항목 | 확인 |
|---|---|
| 키가 서버에만 있나 | `app/page.tsx` 에 `nvapi-` 가 있으면 **실패** |
| 스트리밍이 되나 | 10초 멈췄다 한 번에 나오면 실패 |
| Prisma 스키마가 맞나 | 관계·onDelete 가 빠지지 않았나 |
| `.env` 가 커밋됐나 | `git status` 에 뜨면 **즉시 수정** |

### 원리

**브라우저는 MCP 서버를 못 붙인다.** 그래서 백엔드가 필요하고, 백엔드가 있으니 **키를 숨길 수 있고**, 그래서 **에이전트 루프도 거기서 돈다.**

> 오늘은 **LLM과 대화만** 된다. 내일 여기에 **도구·루프·승인 게이트**를 붙여 진짜 MCP 호스트로 만든다.


## 산출물 체크리스트

**하네스**
- [ ] `/init` 으로 **CLAUDE.md 초안**을 뽑고 5섹션으로 정리했다 (Lab 1)
- [ ] 같은 태스크를 **CLAUDE.md 전/후로 비교**했다
- [ ] `/review` → `+/cso` → `+/qa` 로 **회수율을 3번 측정**했다 (Lab 2)
- [ ] 파이썬으로 **미니 하네스를 직접 만들었다** (Lab 3) — 도구 레지스트리·컨텍스트 트리밍·스킬 로더

**MCP 호스트 뼈대** (Lab 4)
- [ ] Next.js 프로젝트를 **스크래치부터** 만들었다
- [ ] **Neon** DB를 연결하고 `prisma migrate` 로 **실제 테이블**을 만들었다
- [ ] `.env` 에 키를 넣고 **`.gitignore` 에 있는지 확인**했다
- [ ] NVIDIA Qwen을 **서버에서만** 부른다 (프론트에 키가 없다)
- [ ] 응답이 **스트리밍**된다 (글자가 흘러나온다)

> 한 줄: **오늘은 대화만 된다. 내일 도구·루프·승인을 붙여 진짜 호스트로 만든다.**
